In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_sample_weight

# New: target encoding for country_or_area + SMOTE for class imbalance
from category_encoders import TargetEncoder
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb
import shap
import pickle
import os

pd.set_option('display.max_columns', None)

c:\trade_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df = pd.read_parquet("../dataset/trade_cleaned.parquet")
print(df.shape)
df.head()

(7876052, 13)


,country_or_area,year,commodity,flow,trade_usd,weight_kg,quantity_name,quantity,category,category_final,unit_price,price_per_kg,qty_per_kg
0,Afghanistan,2016,"Sheep, live",Export,6088,2339.0,Number of items,51.0,"Food, Beverages & Tobacco",Food & Agri,119.372549,2.602822,0.021804
1,Afghanistan,2016,"Goats, live",Export,3958,984.0,Number of items,53.0,"Food, Beverages & Tobacco",Food & Agri,74.679245,4.022358,0.053862
2,Afghanistan,2008,"Bovine animals, live pure-bred breeding",Import,1026804,272.0,Number of items,3769.0,"Food, Beverages & Tobacco",Food & Agri,272.434067,3775.014706,13.856618
3,Albania,2016,"Bovine animals, live, except pure-bred breeding",Import,2414533,1114023.0,Number of items,6853.0,"Food, Beverages & Tobacco",Food & Agri,352.332263,2.167400,0.006152
4,Albania,2016,"Swine, live except pure-bred breeding > 50 kg",Import,14265937,9484953.0,Number of items,96040.0,"Food, Beverages & Tobacco",Food & Agri,148.541618,1.504060,0.010126


In [6]:
df["category_final"].value_counts()

category_final
Food & Agri                1977773
Textiles & Apparel         1282684
Chemicals & Plastics       1185371
Minerals & Metals          1143222
Precision & Misc            593295
Leather & Footwear          457967
Wood & Paper                412838
Stone & Glass               380028
Transport Equipment         267560
Machinery & Electronics     175314
Name: count, dtype: int64

In [7]:
# Stratified sample (preserves the ~11x class imbalance as-is, we'll handle
# imbalance via SMOTE during training, not via oversampling here)
SAMPLE_FRAC = 0.28   # ~2.2M rows out of 7.87M (bumped up from 0.19 since
                     # we now have a bigger time budget -- more data helps
                     # minority classes especially, before SMOTE even kicks in)

df_sample = (
    df.groupby('category_final', group_keys=False)
      .apply(lambda x: x.sample(frac=SAMPLE_FRAC, random_state=42))
      .reset_index(drop=True)
)
print("Sampled shape:", df_sample.shape)
print(df_sample['category_final'].value_counts(normalize=True))

# Single train/test split, reused for both regression and classification
# (same X rows, different y)
train_df, test_df = train_test_split(
    df_sample, test_size=0.2, random_state=42, stratify=df_sample['category_final']
)
print(train_df.shape, test_df.shape)

C:\Users\arman\AppData\Local\Temp\ipykernel_11640\3926103586.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=SAMPLE_FRAC, random_state=42))


Sampled shape: (2205296, 13)
category_final
Food & Agri                0.251112
Textiles & Apparel         0.162859
Chemicals & Plastics       0.150503
Minerals & Metals          0.145151
Precision & Misc           0.075329
Leather & Footwear         0.058147
Wood & Paper               0.052417
Stone & Glass              0.048251
Transport Equipment        0.033971
Machinery & Electronics    0.022259
Name: proportion, dtype: float64
(1764236, 13) (441060, 13)


In [8]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Encodes high-cardinality categorical cols as their frequency (proportion)
    seen in the TRAINING data. Unseen categories at transform time -> 0."""
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        self.freq_maps_ = {
            col: X[col].value_counts(normalize=True).to_dict()
            for col in self.cols
        }
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col] = X[col].map(self.freq_maps_[col]).fillna(0.0)
        return X[self.cols].values

    def get_feature_names_out(self, input_features=None):
        return np.array([f"{c}_freq" for c in self.cols])

# 1. Regression Model (Trade_usd Prediction)

In [11]:
# Regression setup ----
reg_features = ['country_or_area', 'commodity', 'flow', 'quantity_name',
                 'year', 'weight_kg', 'quantity']

X_train_reg = train_df[reg_features].copy()
X_test_reg  = test_df[reg_features].copy()

y_train_reg = np.log1p(train_df['trade_usd'])
y_test_reg  = np.log1p(test_df['trade_usd'])

freq_cols   = ['country_or_area', 'commodity']
onehot_cols = ['flow', 'quantity_name']
num_cols    = ['year', 'weight_kg', 'quantity']

preprocessor_reg = ColumnTransformer(transformers=[
    ('freq', FrequencyEncoder(cols=freq_cols), freq_cols),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), onehot_cols),
    ('num', 'passthrough', num_cols),
])


# Regression pipeline: XGBoost + fit
reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_reg),
    ('model', xgb.XGBRegressor(
        n_estimators=400,
        max_depth=8,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    ))
])

reg_pipeline.fit(X_train_reg, y_train_reg)


# Regression evaluation (reverse log for reporting)
y_pred_log = reg_pipeline.predict(X_test_reg)

# Reverse log1p to report metrics on ORIGINAL trade_usd scale
y_test_actual = np.expm1(y_test_reg)
y_pred_actual = np.expm1(y_pred_log)
y_pred_actual = np.clip(y_pred_actual, 0, None)  # safety: no negative $

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae  = mean_absolute_error(y_test_actual, y_pred_actual)
r2   = r2_score(y_test_actual, y_pred_actual)

print(f"RMSE (USD): {rmse:,.2f}")
print(f"MAE  (USD): {mae:,.2f}")
print(f"R²        : {r2:.4f}")

# Also worth reporting R² on log scale (more stable, less skewed by outliers)
r2_log = r2_score(y_test_reg, y_pred_log)
print(f"R² (log scale): {r2_log:.4f}")

RMSE (USD): 694,441,217.75
MAE  (USD): 16,550,744.26
R²        : 0.1079
R² (log scale): 0.8584


# 2. Classification Model (Category Prediction)

In [9]:
clf_features = ['country_or_area', 'commodity', 'flow', 'quantity_name',
                'year', 'weight_kg', 'quantity',
                'unit_price', 'price_per_kg', 'qty_per_kg']

train_df['trade_usd_log'] = np.log1p(train_df['trade_usd'])
test_df['trade_usd_log']  = np.log1p(test_df['trade_usd'])
clf_features_full = clf_features + ['trade_usd_log']

X_train_clf = train_df[clf_features_full].copy()
X_test_clf  = test_df[clf_features_full].copy()

print(train_df['category_final'].value_counts())

le = LabelEncoder()
y_train_clf = le.fit_transform(train_df['category_final'])
y_test_clf  = le.transform(test_df['category_final'])

freq_cols_clf   = ['commodity']
target_cols_clf = ['country_or_area']
onehot_cols_clf = ['flow', 'quantity_name']
num_cols_clf    = ['year', 'weight_kg', 'quantity', 'trade_usd_log',
                   'unit_price', 'price_per_kg', 'qty_per_kg']

preprocessor_clf = ColumnTransformer(transformers=[
    ('freq', FrequencyEncoder(cols=freq_cols_clf), freq_cols_clf),
    ('target', TargetEncoder(cols=target_cols_clf, smoothing=10), target_cols_clf),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), onehot_cols_clf),
    ('num', 'passthrough', num_cols_clf),
])

# Small model: 200 rounds, depth 8  ->  tens of MB instead of 216 MB
clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_clf),
    ('model', xgb.XGBClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_weight=5,
        gamma=0.1,
        reg_alpha=0.1,
        tree_method='hist',
        objective='multi:softprob',
        num_class=len(le.classes_),
        random_state=42,
        n_jobs=-1,
        eval_metric='mlogloss'
    ))
])

# class imbalance handled with weights instead of SMOTE (much faster)
sample_w = compute_sample_weight(class_weight='balanced', y=y_train_clf)
clf_pipeline.fit(X_train_clf, y_train_clf, model__sample_weight=sample_w)

# ============ Evaluation ============
y_pred_clf = clf_pipeline.predict(X_test_clf)
print(f"Accuracy : {accuracy_score(y_test_clf, y_pred_clf):.4f}")
print(f"Macro-F1 : {f1_score(y_test_clf, y_pred_clf, average='macro'):.4f}")
print("\n", classification_report(y_test_clf, y_pred_clf, target_names=le.classes_))

# ============ Save + size check ============
import os
with open('../models/classifier_small.pkl', 'wb') as f:
    pickle.dump(clf_pipeline, f)
print("Size (MB):", round(os.path.getsize('../models/classifier_small.pkl') / 1e6, 1))

category_final
Food & Agri                443021
Textiles & Apparel         287321
Chemicals & Plastics       265523
Minerals & Metals          256082
Precision & Misc           132898
Leather & Footwear         102585
Wood & Paper                92476
Stone & Glass               85126
Transport Equipment         59934
Machinery & Electronics     39270
Name: count, dtype: int64
Accuracy : 0.5378
Macro-F1 : 0.5612

                          precision    recall  f1-score   support

   Chemicals & Plastics       0.63      0.40      0.49     66381
            Food & Agri       0.74      0.49      0.59    110755
     Leather & Footwear       0.48      0.77      0.59     25646
Machinery & Electronics       0.77      0.99      0.86      9818
      Minerals & Metals       0.50      0.42      0.46     64020
       Precision & Misc       0.56      0.65      0.60     33225
          Stone & Glass       0.28      0.72      0.41     21282
     Textiles & Apparel       0.62      0.50      0.55     7

In [12]:
import pickle
import os

os.makedirs('../models', exist_ok=True)

with open('../models/regressor.pkl', 'wb') as f:
    pickle.dump(reg_pipeline, f)

with open('../models/classifier.pkl', 'wb') as f:
    pickle.dump(clf_pipeline, f)

with open('../models/label_encoder.pkl', 'wb') as f:  
    pickle.dump(le, f)

print("Models saved successfully")

Models saved successfully
